In [34]:
import pymupdf4llm
from pathlib import Path
from glob import glob
from pprint import pprint

data_dir = "../data"
chunk_size = 500

all_chunks = []
pdf_files = sorted(glob(f"{data_dir}/*.pdf"))
for pdf_path in pdf_files:
    md_text = pymupdf4llm.to_markdown(pdf_path)
    arxiv_id = Path(pdf_path).stem
    for i in range(0, len(md_text), chunk_size):
        all_chunks.append({"text": md_text[i : i + chunk_size], "arxiv_id": arxiv_id})

print(f"Parsed {len(pdf_files)} papers, {len(all_chunks)} chunks total")

Parsed 11 papers, 3556 chunks total


In [35]:
all_chunks[0]

{'text': '# **BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding** \n\n**Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova** Google AI Language \n\n_{_ jacobdevlin,mingweichang,kentonl,kristout _}_ @google.com \n\n## **Abstract** \n\nWe introduce a new language representation model called **BERT** , which stands for **B** idirectional **E** ncoder **R** epresentations from **T** ransformers. Unlike recent language representation models (Peters et al., 2018a; Radford et al., 2', 'arxiv_id': '1810.04805'}

In [36]:
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import PointStruct

client = QdrantClient(url="http://localhost:6333")

collection_name = "arxiv_papers"
embedding_model_dimensions = 384

dense_model = "BAAI/bge-small-en"
sparse_model = "qdrant/bm25"

In [37]:
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense_vector": models.VectorParams(
                size=embedding_model_dimensions, distance=models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "bm25_sparse_vector": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )
        },
    )

In [38]:
# client.delete_collection(collection_name=collection_name)

In [41]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

texts = [c["text"] for c in all_chunks]

dense_encoder = SentenceTransformer(dense_model)
dense_embeddings = dense_encoder.encode(texts, show_progress_bar=True)

bm25_encoder = SparseTextEmbedding(model_name=sparse_model)
sparse_embeddings = list(bm25_encoder.embed(texts))

print(f"Dense shape: {dense_embeddings.shape}")
print(f"Sparse vectors: {len(sparse_embeddings)}")

Batches: 100%|██████████| 112/112 [01:10<00:00,  1.58it/s]


Dense shape: (3556, 384)
Sparse vectors: 3556


In [42]:
points = []
for idx, (chunk, dense_vec, sparse_vec) in enumerate(
    zip(all_chunks, dense_embeddings, sparse_embeddings)
):
    point = PointStruct(
        id=idx + 1,
        payload={
            "text": chunk["text"],
            "arxiv_id": chunk["arxiv_id"],
            "chunk_idx": idx,
        },
        vector={
            "dense_vector": dense_vec.tolist(),
            "bm25_sparse_vector": models.SparseVector(
                indices=sparse_vec.indices.tolist(), values=sparse_vec.values.tolist()
            ),
        },
    )
    points.append(point)

client.upload_points(collection_name=collection_name, points=points, batch_size=64)
print(f"Uploaded {len(points)} chunks from {len(pdf_files)} papers")

Uploaded 3556 chunks from 11 papers


In [50]:
query = "which company introduced LLAMA"

dense_query_vec = dense_encoder.encode(query)
sparse_query_vec = next(bm25_encoder.embed([query]))

In [51]:
results = client.query_points(
    collection_name=collection_name,
    prefetch=[
        models.Prefetch(query=dense_query_vec.tolist(), using="dense_vector", limit=5),
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query_vec.indices.tolist(),
                values=sparse_query_vec.values.tolist(),
            ),
            using="bm25_sparse_vector",
            limit=5,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
)


pprint(results.points)

[ScoredPoint(id=3158, version=50, score=0.7, payload={'text': 'h/llama` . Finally, we are sharing a _Responsible Use Guide_ , which provides guidelines regarding safe development and deployment. \n\n**Responsible Release.** While many companies have opted to build AI behind closed doors, we are releasing Llama 2 openly to encourage responsible AI innovation. Based on our experience, an open approach draws upon the collective wisdom, diversity, and ingenuity of the AI-practitioner community to realize the benefits of this technology. Collaboration will make th', 'arxiv_id': '2307.09288', 'chunk_idx': 3157}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3471, version=55, score=0.5, payload={'text': ' 52: Model card for Llama 2.** \n\n77 \n\n', 'arxiv_id': '2307.09288', 'chunk_idx': 3470}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2946, version=47, score=0.33333334, payload={'text': ' summarizes the carbon emission for pretraining the Llama 2 family